In [1]:
# =========================
# Standard library imports
# =========================
import sys
import os
import io
from pathlib import Path
import math
import json
import pickle
import logging
import random
import subprocess
import warnings
import collections
import itertools
import re
from IPython.display import display

warnings.filterwarnings("ignore")


# =========================
# Numeric / stats
# =========================
import numpy as np
import pandas as pd
from scipy import sparse
from scipy import stats
from scipy.stats import rankdata
from scipy import __version__ as scipy_version


# =========================
# Plotting / visualization
# =========================
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, FormatStrFormatter, MaxNLocator
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
import seaborn as sns


# =========================
# scikit-learn
# =========================
from sklearn import model_selection, metrics
from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit,
    KFold,
    GroupKFold,
    cross_val_score,
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neighbors import LocalOutlierFactor
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
)
from sklearn import __version__ as sklearn_version


# =========================
# Optimization / AutoML
# =========================
import optuna


# =========================
# LightGBM
# =========================
try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
    _HAS_LGBM = True
except Exception as e:
    _HAS_LGBM = False
    raise RuntimeError(
        "LightGBM not installed. Please install it with `pip install lightgbm`."
    ) from e


# =========================
# RNA structure (ViennaRNA)
# =========================
import RNA


# =========================
# UpSet plots
# =========================
try:
    from upsetplot import UpSet, from_memberships
except Exception:
    _ = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "upsetplot"],
        check=False,
    )
    from upsetplot import UpSet, from_memberships


# =========================
# Model explanation
# =========================
import shap


# =========================
# User-provided utilities
# =========================
import sylib  


# =========================
# Version info
# =========================
print(f"python    = {sys.version_info[0]}.{sys.version_info[1]}.{sys.version_info[2]}")
print(f"pandas    = {pd.__version__}")
print(f"numpy     = {np.__version__}")
print(f"scipy     = {scipy_version}")
print(f"optuna    = {optuna.__version__}")
print(f"sklearn   = {sklearn_version}")
print(f"ViennaRNA = {RNA.__version__}")
print(f"lightgbm  = {lgb.__version__}")
print(f"sylib     = {sylib.__version__}")


# =========================
# Progress bar & logging
# =========================
# progress bar from sylib
progress_bar = sylib.utils.ProgressBar()

# reset handlers then configure logging
logging.root.handlers = []
stream_handler = logging.StreamHandler(sys.stderr)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)8s: %(message)s",
    handlers=[stream_handler],
)
logger = logging.getLogger(__name__)

# make matplotlib quieter
logging.getLogger("matplotlib").setLevel(logging.WARNING)

python    = 3.11.15
pandas    = 2.3.3
numpy     = 2.4.6
scipy     = 1.17.1
optuna    = 4.9.0
sklearn   = 1.9.0
ViennaRNA = 2.7.2
lightgbm  = 4.6.0
sylib     = 0.3.0.dev0+ae18bb2


In [2]:
from pathlib import Path

RAW = Path("../data/raw")
files = sorted(RAW.glob("*.tsv.gz"))

for f in files:
    print(f.name)

AT21.PR_var.N50-TSL0-ARL0.MFE.tsv.gz
AT21.PR_var.N50-TSL0-ARL0.RNA.seq_data.tsv.gz
AT21.PR_var.dev_data.MFE.tsv.gz
AT21.PR_var.dev_data.RNA.seq_data.tsv.gz
NB21.PR_var.N50-TSL0-ARL0.MFE.tsv.gz
NB21.PR_var.N50-TSL0-ARL0.RNA.seq_data.tsv.gz
NB21.PR_var.dev_data.MFE.tsv.gz
NB21.PR_var.dev_data.RNA.seq_data.tsv.gz
OS21.PR_var.N50-TSL0-ARL0.MFE.tsv.gz
OS21.PR_var.N50-TSL0-ARL0.RNA.seq_data.tsv.gz
OS21.PR_var.dev_data.MFE.tsv.gz
OS21.PR_var.dev_data.RNA.seq_data.tsv.gz


In [3]:
DATA_DIR = Path("../data/raw")

score_col_name = "PR_var"
group_col_name = None

rna_temp = 22

test_file = DATA_DIR / "AT21.PR_var.N50-TSL0-ARL0.RNA.seq_data.tsv.gz"

seq_df, metadata = sylib.fileio.load_df(
    str(test_file)
)

metadata.print_minimum_data(
    label="Seq data",
    logger=logger,
    logging_level="info"
)

display(seq_df)

2026-06-16 16:32:29,314     INFO: Seq data |   name = AT21.PR_var.N50-TSL0-ARL0.RNA.seq_data.tsv.gz
2026-06-16 16:32:29,315     INFO: Seq data |    md5 = d66042fbcc8c89972a7d978665f0d63f
2026-06-16 16:32:29,316     INFO: Seq data | md5_gz = 32f956b63477c6726ca680a8317ad7e9


,var_id,trans_id,gene_id,usage,PR_var,5'UTR,CDS,3'UTR,n_5'UTR_introns,n_CDS_introns,n_3'UTR_introns
0,AT1G01050.1.33113.31207,AT1G01050.1,AT1G01050,0.202151,0.798371,GGGCUUCACUUGUUUCUCUCCCAAAGUUUCUUCAUCAUCCUUGCGA...,AUGAGUGAAGAAACUAAAGAUAACCAGAGGCUGCAGCGACCAGCUC...,AGCUUCUCCUCAGAAGAUUUCUGCAGCAUCUAUGUUUCUGUUACUU...,1,7,0
1,AT1G01100.2.51179.50115,AT1G01100.2,AT1G01100,0.010849,0.787718,AGAAAUUUUGUCUCUCUCGCCGCCUUGCGAAAAGCAUUUUCGAUCU...,AUGUCGACAGUUGGAGAGCUUGCUUGCAGCUACGCUGUUAUGAUCC...,ACGCAGUCACUUGUCUUUCUUCUUUGUAGUUGGAUAUUGGAGACUA...,1,2,0
2,AT1G01100.2.51181.50091,AT1G01100.2,AT1G01100,0.036703,0.449890,CUAGAAAUUUUGUCUCUCUCGCCGCCUUGCGAAAAGCAUUUUCGAU...,AUGUCGACAGUUGGAGAGCUUGCUUGCAGCUACGCUGUUAUGAUCC...,ACGCAGUCACUUGUCUUUCUUCUUUGUAGUUGGAUAUUGGAGACUA...,1,2,0
3,AT1G01100.2.51181.50094,AT1G01100.2,AT1G01100,0.029799,0.490931,CUAGAAAUUUUGUCUCUCUCGCCGCCUUGCGAAAAGCAUUUUCGAU...,AUGUCGACAGUUGGAGAGCUUGCUUGCAGCUACGCUGUUAUGAUCC...,ACGCAGUCACUUGUCUUUCUUCUUUGUAGUUGGAUAUUGGAGACUA...,1,2,0
4,AT1G01100.2.51181.50115,AT1G01100.2,AT1G01100,0.087002,0.565213,CUAGAAAUUUUGUCUCUCUCGCCGCCUUGCGAAAAGCAUUUUCGAU...,AUGUCGACAGUUGGAGAGCUUGCUUGCAGCUACGCUGUUAUGAUCC...,ACGCAGUCACUUGUCUUUCUUCUUUGUAGUUGGAUAUUGGAGACUA...,1,2,0
...,...,...,...,...,...,...,...,...,...,...,...
7166,AT5G67600.1.26960273.26959591,AT5G67600.1,AT5G67600,0.017988,0.771029,AAAGAGAGAAAGAAAAAAAAGCUCAAAGAACUCAGAAGAAGAAGAAAG,AUGUAUCAUCAAGAACAACAUCCUGUCGGUGCUCCUCCUCCCCAAG...,UCGGAGGAUAAUUAUAUACGUUGCCUCAGUCAACUUUUAUUUCCUC...,0,2,0
7167,AT5G67600.1.26960273.26959615,AT5G67600.1,AT5G67600,0.026138,0.939605,AAAGAGAGAAAGAAAAAAAAGCUCAAAGAACUCAGAAGAAGAAGAAAG,AUGUAUCAUCAAGAACAACAUCCUGUCGGUGCUCCUCCUCCCCAAG...,UCGGAGGAUAAUUAUAUACGUUGCCUCAGUCAACUUUUAUUUCCUC...,0,2,0
7168,AT5G67600.1.26960273.26959630,AT5G67600.1,AT5G67600,0.078977,0.775602,AAAGAGAGAAAGAAAAAAAAGCUCAAAGAACUCAGAAGAAGAAGAAAG,AUGUAUCAUCAAGAACAACAUCCUGUCGGUGCUCCUCCUCCCCAAG...,UCGGAGGAUAAUUAUAUACGUUGCCUCAGUCAACUUUUAUUUCCUC...,0,2,0
7169,AT5G67600.1.26960277.26959630,AT5G67600.1,AT5G67600,0.023328,0.755547,AAACAAAGAGAGAAAGAAAAAAAAGCUCAAAGAACUCAGAAGAAGA...,AUGUAUCAUCAAGAACAACAUCCUGUCGGUGCUCCUCCUCCCCAAG...,UCGGAGGAUAAUUAUAUACGUUGCCUCAGUCAACUUUUAUUUCCUC...,0,2,0


In [ ]:
# Fucking feature evaluation, this is part of you to make you algorithm/functions to shit everything.